# XCB (Corecoin) Mining via Tor Network

Anonymous CPU mining using tor-proxy-miner-amd64

## Configuration

Edit the variables below before running:

In [ ]:
# CONFIGURATION - EDIT THESE VALUES
POOL_HOST = "dach.corecoinpool.com"  # Pool hostname
POOL_PORT = "3333"                    # Pool port
WALLET_ADDRESS = "CB..."              # Your Corecoin wallet address
THREADS = None                        # None = auto-detect all cores
TOR_PORT = 9050                       # Local Tor SOCKS5 port
USE_WORKER = False                    # False = all VPS combined, True = unique worker per VPS

## Step 1: Install Dependencies

In [ ]:
!apt update
!apt install -y git cmake build-essential libssl-dev libhwloc-dev wget

## Step 2: Download tor-proxy-miner-amd64

In [ ]:
import os

# Download tor-proxy-miner binary
!wget -O /tmp/tor-proxy-miner-amd64 https://github.com/catchthatrabbit/tor-proxy-miner/releases/latest/download/tor-proxy-miner-amd64
!chmod +x /tmp/tor-proxy-miner-amd64
!mv /tmp/tor-proxy-miner-amd64 /usr/local/bin/

print("\n✅ tor-proxy-miner-amd64 installed!")

## Step 3: Clone and Compile CoreMiner

This will take 5-10 minutes depending on CPU power.

In [ ]:
import os

# Clone repository
!git clone https://github.com/catchthatrabbit/coreminer.git /tmp/coreminer

# Compile
os.makedirs('/tmp/coreminer/build', exist_ok=True)
os.chdir('/tmp/coreminer/build')

!cmake ..
!make -j$(nproc)

# Install
!cp /tmp/coreminer/build/coreminer /usr/local/bin/
!chmod +x /usr/local/bin/coreminer

print("\n✅ CoreMiner compiled and installed successfully!")

## Step 4: Start Tor Proxy

This cell starts Tor proxy in background. Wait 30 seconds for bootstrap.

In [ ]:
import subprocess
import time

# Start Tor proxy in background
tor_process = subprocess.Popen(
    ['/usr/local/bin/tor-proxy-miner-amd64', '--port', str(TOR_PORT)],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print(f"Starting Tor proxy on port {TOR_PORT}...")
print("Waiting 30 seconds for Tor to bootstrap...")
time.sleep(30)
print("\n✅ Tor proxy ready!")

## Step 5: Generate Random Worker Name

In [ ]:
import random\nimport string\n\ndef generate_worker_name():\n    prefix = random.choice(['compute', 'node', 'rig', 'proc', 'worker', 'cpu'])\n    suffix = ''.join(random.choices(string.ascii_lowercase + string.digits, k=8))\n    return f\"{prefix}{suffix}\"\n\nif USE_WORKER:\n    WORKER_NAME = generate_worker_name()\n    print(f\"Worker Name: {WORKER_NAME}\")\nelse:\n    WORKER_NAME = None\n    print(\"Worker Name: None (all VPS combined into one worker)\")

## Step 6: Start Mining via Tor

**Note:** This cell will run continuously. Press the ⬛ (stop) button to stop mining.

In [ ]:
import subprocess
import os

# Auto-detect threads if not specified
if THREADS is None:
    THREADS = os.cpu_count() or 4

# Build connection string (through Tor SOCKS5 proxy)\npool_url = f\"socks5://127.0.0.1:{TOR_PORT}@{POOL_HOST}:{POOL_PORT}\"\n\nif USE_WORKER:\n    worker_id = f\"{WALLET_ADDRESS}.{WORKER_NAME}\"\n    worker_display = WORKER_NAME\nelse:\n    worker_id = WALLET_ADDRESS\n    worker_display = \"None (combined hashrate)\"\n\nprint(f\"{'='*60}\")\nprint(f\"Starting XCB Mining via Tor\")\nprint(f\"{'='*60}\")\nprint(f\"Pool: {POOL_HOST}:{POOL_PORT}\")\nprint(f\"Tor Proxy: 127.0.0.1:{TOR_PORT}\")\nprint(f\"Worker: {worker_display}\")\nprint(f\"Threads: {THREADS}\")\nprint(f\"{'='*60}\\n\")\n
# Start mining
subprocess.run([
    '/usr/local/bin/coreminer',
    '-o', pool_url,
    '-u', worker_id,
    '-t', str(THREADS)
])

## Optional: Check Mining Status

Run this in a separate cell while mining:

In [ ]:
# Check if miner is running
!ps aux | grep coreminer | grep -v grep

# Check Tor proxy
!ps aux | grep tor-proxy-miner | grep -v grep

## Stop Mining and Tor

If you need to stop everything:

In [ ]:
!pkill -9 coreminer
!pkill -9 tor-proxy-miner-amd64
print("Mining and Tor proxy stopped.")

## Check Current Tor Exit IP

Verify your mining traffic is going through Tor:

In [ ]:
import requests

# Check IP through Tor proxy
proxies = {
    'http': f'socks5://127.0.0.1:{TOR_PORT}',
    'https': f'socks5://127.0.0.1:{TOR_PORT}'
}

try:
    response = requests.get('https://api.ipify.org?format=json', proxies=proxies, timeout=10)
    print(f"Current Tor Exit IP: {response.json()['ip']}")
except Exception as e:
    print(f"Could not get Tor IP: {e}")